# ProtoLM for Drug Review Classification

Using Proto-LM to analyze the drug reviews dataset for 10-class rating prediction.

**References:**
- [Proto-LM GitHub](https://github.com/yx131/proto-lm/tree/main)
- [Drug Review Dataset](https://www.kaggle.com/datasets/mohamedabdelwahabali/drugreview/data)

## Imports and Setup

In [1]:
import argparse
!pip install pytorch-lightning
!pip install -U datasets
import torch
from torch.utils.data import DataLoader
import pytorch_lightning as pl
from transformers import AutoConfig, AutoTokenizer, AutoModelForSequenceClassification
from ProtoLM import proto_lm
from proto_data_class import sst_datamodule
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger
import datasets
from tqdm import tqdm
import numpy as np
import torch.nn.functional as F
from collections import Counter

## Configuration and Data Setup

In [2]:
# Model configuration
model_name = 'bert-base-uncased'

# Get model configuration
config = AutoConfig.from_pretrained(model_name)
hidden_size = config.hidden_size  # 768 for bert-base-uncased

# ProtoLM parameters - following tutorial pattern
args = {
    'model_name': model_name,
    'max_seq_length': 100,
    'num_prototypes': 200,      # Tutorial uses 200
    'hidden_shape': hidden_size,
    'num_classes': 10,          # 10-class classification for ratings 1-10
    'cohsep_ratio': 0.5,        # Tutorial default
    'lambda0': 0.1,             # Tutorial default
    'lr': 1e-4,                 # Standard learning rate
    'proto_training_weights': 1,
    'batch_size': 128,
    'max_epochs': 10,
}

print(f"Configuration: {args}")

Configuration: {'model_name': 'bert-base-uncased', 'max_seq_length': 100, 'num_prototypes': 200, 'hidden_shape': 768, 'num_classes': 10, 'cohsep_ratio': 0.5, 'lambda0': 0.1, 'lr': 0.0001, 'proto_training_weights': 1, 'batch_size': 128, 'max_epochs': 10}


In [3]:
# Setup data module
drug_review_dm = sst_datamodule(
    model_name_or_path=args['model_name'],
    max_seq_length=args['max_seq_length'],
    train_batch_size=args['batch_size'],
    eval_batch_size=args['batch_size']
)
drug_review_dm.setup(stage='fit')

# Load base model
base_model = AutoModelForSequenceClassification.from_pretrained(
    args['model_name'],
    num_labels=args['num_classes'],
    ignore_mismatched_sizes=True
)

if hasattr(base_model, "roberta"):
    llm_model = base_model.roberta
elif hasattr(base_model, "bert"):
    llm_model = base_model.bert
else:
    llm_model = base_model

print("Data module and base model loaded successfully")

Loading dataset from local files...
🔍 Detected local development environment
📁 Looking for data files in: c:\Users\bar24\OneDrive - Universiteit Utrecht\Documents\School\UU Data Sceince MSc\1st Year\Period 4\Human centered machine learning (INFOMHCML)\Project\HCML-NLP-Project\Data
✅ All data files found!
split is: train
self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']
split is: validation
split is: train
self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']
split is: validation


Map:   0%|          | 0/27703 [00:00<?, ? examples/s]

self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']
split is: test


Map:   0%|          | 0/46108 [00:00<?, ? examples/s]

self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Data module and base model loaded successfully


## ProtoLM Model Definition

In [4]:
class DrugReviewProtoLM(proto_lm):
    """ProtoLM adapted for drug review classification following the tutorial pattern."""
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        print(f"DrugReviewProtoLM initialized:")
        print(f"  Prototypes: {self.hparams.num_prototypes}")
        print(f"  Classes: {self.hparams.num_classes}")
        print(f"  Lambda0: {self.hparams.lambda0}")
        print(f"  Cohsep ratio: {self.hparams.cohsep_ratio}")
    
    def training_step(self, batch, batch_idx):
        """Training step following the tutorial pattern"""
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        
        # Handle sentiment features
        if "sentiment_features" in batch:
            sentiment_features = batch["sentiment_features"]
        else:
            batch_size = input_ids.shape[0]
            sentiment_features = torch.zeros(batch_size, 4, device=input_ids.device)
        
        # Forward pass
        outputs = self(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features,
            labels=labels
        )
        
        # Calculate loss
        similarities = outputs['similarities']
        logits = outputs['logits']
        all_losses = self.calc_loss(logits, labels, similarities)
        total_loss = all_losses['total_loss']
        
        # Calculate accuracy
        preds = torch.argmax(outputs['probs'], dim=1)
        accuracy = (preds == labels).float().mean()
        
        # Log metrics
        self.log('train_loss', total_loss, prog_bar=True, on_step=True, on_epoch=True)
        self.log('train_accuracy', accuracy, prog_bar=True, on_step=True, on_epoch=True)
        self.log('train_ce_loss', all_losses['ce_loss'], on_step=False, on_epoch=True)
        self.log('train_cohesion_loss', all_losses['cohesion_loss'], on_step=False, on_epoch=True)
        self.log('train_separation_loss', all_losses['separation_loss'], on_step=False, on_epoch=True)
        
        return total_loss
    
    def validation_step(self, batch, batch_idx):
        """Validation step following the tutorial pattern"""
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["labels"]
        
        # Handle sentiment features
        if "sentiment_features" in batch:
            sentiment_features = batch["sentiment_features"]
        else:
            batch_size = input_ids.shape[0]
            sentiment_features = torch.zeros(batch_size, 4, device=input_ids.device)
        
        # Forward pass
        outputs = self(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features,
            labels=labels
        )
        
        # Calculate loss
        similarities = outputs['similarities']
        logits = outputs['logits']
        all_losses = self.calc_loss(logits, labels, similarities)
        val_loss = all_losses['total_loss']
        
        # Calculate accuracy
        preds = torch.argmax(outputs['probs'], dim=1)
        accuracy = (preds == labels).float().mean()
        
        # Log metrics
        self.log('val_loss', val_loss, prog_bar=True, on_epoch=True)
        self.log('val_accuracy', accuracy, prog_bar=True, on_epoch=True)
        
        return {'val_loss': val_loss, 'val_accuracy': accuracy}

# Create the model
proto = DrugReviewProtoLM(
    pretrained_model=llm_model,
    max_seq_length=args['max_seq_length'],
    num_prototypes=args['num_prototypes'],
    hidden_shape=args['hidden_shape'],
    num_classes=args['num_classes'],
    cohsep_ratio=args['cohsep_ratio'],
    lambda0=args['lambda0'],
    lr=args['lr'],
    proto_training_weights=bool(args['proto_training_weights'])
)

print("ProtoLM model created successfully")

DrugReviewProtoLM initialized:
  Prototypes: 200
  Classes: 10
  Lambda0: 0.1
  Cohsep ratio: 0.5
ProtoLM model created successfully


## Model Diagnostics

In [5]:
# Test forward pass
print("Testing ProtoLM forward pass...")

test_batch = next(iter(drug_review_dm.train_dataloader()))
device = 'cuda' if torch.cuda.is_available() else 'cpu'
proto = proto.to(device)

try:
    proto.eval()
    with torch.no_grad():
        # Small test batch
        input_ids = test_batch['input_ids'][:4].to(device)
        attention_mask = test_batch['attention_mask'][:4].to(device)
        labels = test_batch['labels'][:4].to(device)
        
        if 'sentiment_features' in test_batch:
            sentiment_features = test_batch['sentiment_features'][:4].to(device)
        else:
            sentiment_features = torch.zeros(4, 4, device=device)
        
        # Forward pass
        outputs = proto(
            input_ids=input_ids,
            attention_mask=attention_mask,
            sentiment_features=sentiment_features
        )
        
        print(f"Forward pass successful!")
        print(f"  Logits shape: {outputs['probs'].shape}")
        print(f"  Similarities shape: {outputs['similarities'].shape}")
        
        # Test loss calculation
        all_losses = proto.calc_loss(outputs['logits'], labels, outputs['similarities'])
        print(f"Loss calculation successful!")
        print(f"  CE loss: {all_losses['ce_loss'].item():.4f}")
        print(f"  Cohesion loss: {all_losses['cohesion_loss'].item():.4f}")
        print(f"  Separation loss: {all_losses['separation_loss'].item():.4f}")
        print(f"  Total loss: {all_losses['total_loss'].item():.4f}")
        
        # Check predictions
        preds = torch.argmax(outputs['probs'], dim=1)
        print(f"Predictions: {preds.cpu().tolist()}")
        print(f"Labels: {labels.cpu().tolist()}")
        print(f"Unique predictions: {torch.unique(preds).cpu().tolist()}")
        
        print("ProtoLM is ready for training!")
        
except Exception as e:
    print(f"Forward pass failed: {str(e)}")
    import traceback
    traceback.print_exc()

proto.train()

Testing ProtoLM forward pass...
returned self batch size: 128
Forward pass successful!
  Logits shape: torch.Size([4, 10])
  Similarities shape: torch.Size([4, 200])
Loss calculation successful!
  CE loss: 0.2321
  Cohesion loss: 0.0079
  Separation loss: -0.0085
  Total loss: 0.2315
Predictions: [1, 1, 1, 1]
Labels: [0, 8, 9, 7]
Unique predictions: [1]
ProtoLM is ready for training!
Forward pass successful!
  Logits shape: torch.Size([4, 10])
  Similarities shape: torch.Size([4, 200])
Loss calculation successful!
  CE loss: 0.2321
  Cohesion loss: 0.0079
  Separation loss: -0.0085
  Total loss: 0.2315
Predictions: [1, 1, 1, 1]
Labels: [0, 8, 9, 7]
Unique predictions: [1]
ProtoLM is ready for training!


DrugReviewProtoLM(
  (LLM): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elementw

## Training Setup

In [6]:
# Setup trainer following tutorial pattern
from pytorch_lightning.callbacks import ModelCheckpoint
from pytorch_lightning.loggers.tensorboard import TensorBoardLogger

# Logger
logger = TensorBoardLogger('tb_logs', name='drug_review_proto_lm')

# Checkpoint callback
checkpoint_callback = ModelCheckpoint(
    dirpath='ckpt_dir/drug_review_config',
    monitor='val_loss',
    save_top_k=3,
    filename="{epoch}-{val_loss:.4f}-{val_accuracy:.4f}",
    save_last=True,
    verbose=True
)

# Trainer
trainer = pl.Trainer(
    max_epochs=args['max_epochs'],
    accelerator="auto",
    devices=1,
    logger=logger,
    callbacks=[checkpoint_callback],
    precision=32,
    enable_progress_bar=True,
    log_every_n_steps=50,
    val_check_interval=1.0
)

print("Trainer setup complete")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
`Trainer(val_check_interval=1.0)` was configured so validation will run at the end of the training epoch..


Trainer setup complete


## Training

In [7]:
# Train the model
print("Starting ProtoLM training...")

try:
    trainer.fit(proto, datamodule=drug_review_dm)
    print("Training completed!")
    
    # Get final metrics
    if hasattr(trainer, 'callback_metrics'):
        final_val_acc = trainer.callback_metrics.get('val_accuracy', 0)
        final_val_loss = trainer.callback_metrics.get('val_loss', float('inf'))
        print(f"Final validation accuracy: {final_val_acc:.4f}")
        print(f"Final validation loss: {final_val_loss:.4f}")
        
        if final_val_acc > 0.15:
            print("Training successful - model is learning!")
        else:
            print("Low accuracy - may need hyperparameter tuning")
    
except Exception as e:
    print(f"Training failed: {str(e)}")
    import traceback
    traceback.print_exc()

You are using a CUDA device ('NVIDIA GeForce RTX 3050 Ti Laptop GPU') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Starting ProtoLM training...
split is: train


Map:   0%|          | 0/110811 [00:00<?, ? examples/s]

c:\Users\bar24\OneDrive - Universiteit Utrecht\Documents\School\UU Data Sceince MSc\1st Year\Period 4\Human centered machine learning (INFOMHCML)\Project\HCML-NLP-Project\src\proto_lm\proto_data_class.py:168: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sentiment_neg = torch.tensor(example_batch["sentiment_neg"], dtype=torch.float32)
c:\Users\bar24\OneDrive - Universiteit Utrecht\Documents\School\UU Data Sceince MSc\1st Year\Period 4\Human centered machine learning (INFOMHCML)\Project\HCML-NLP-Project\src\proto_lm\proto_data_class.py:169: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  sentiment_neu = torch.tensor(example_batch["sentiment_neu"], dtype=torch.float32)
c:\Users\bar24\OneDriv

self.columns: ['sentiment_neg', 'sentiment_neu', 'sentiment_pos', 'sentiment_compound', 'input_ids', 'token_type_ids', 'attention_mask', 'labels', 'sentiment_features']
split is: validation


...

c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=15` in the `DataLoader` to improve performance.


Training: |          | 0/? [00:00<?, ?it/s]


Detected KeyboardInterrupt, attempting graceful shutdown ...


Training failed: name 'exit' is not defined


Traceback (most recent call last):
  File "c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\pytorch_lightning\trainer\call.py", line 48, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\pytorch_lightning\trainer\trainer.py", line 599, in _fit_impl
    self._run(model, ckpt_path=ckpt_path)
  File "c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\pytorch_lightning\trainer\trainer.py", line 1012, in _run
    results = self._run_stage()
              ^^^^^^^^^^^^^^^^^
  File "c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\pytorch_lightning\trainer\trainer.py", line 1056, in _run_stage
    self.fit_loop.run()
  File "c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\pytorch_lightning\loops\fit_loop.py", line 216, in run
    self.advance()
  File "c:\Users\bar24\anaconda3\envs\ExplainableAI\Lib\site-packages\pyto

## Model Evaluation

In [ ]:
# Evaluation utilities
class RatingPredictionWrapper:
    """Wrapper to convert ProtoLM 0-9 class predictions to 1-10 drug ratings."""
    
    def __init__(self, model):
        self.model = model
        self.model.eval()

    def predict_rating(self, input_ids, attention_mask, sentiment_features):
        """Predict drug rating (1-10) from model output."""
        with torch.no_grad():
            outputs = self.model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                sentiment_features=sentiment_features
            )

            probs = torch.softmax(outputs['probs'], dim=1)
            class_preds = torch.argmax(probs, dim=1)
            ratings = class_preds + 1  # Convert 0-9 to 1-10
            confidence = torch.max(probs, dim=1)[0]

            return (
                ratings.cpu().numpy(),
                probs.cpu().numpy(),
                confidence.cpu().numpy()
            )

def evaluate_model(model, data_module, device='cpu'):
    """Comprehensive model evaluation."""
    from sklearn.metrics import accuracy_score, classification_report
    
    wrapper = RatingPredictionWrapper(model)
    test_dataloader = data_module.test_dataloader()

    all_ratings = []
    all_true_ratings = []
    all_class_preds = []
    all_true_classes = []

    for batch in tqdm(test_dataloader, desc="Evaluating"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        sentiment_features = batch["sentiment_features"].to(device)
        true_classes = batch["labels"].cpu().numpy()

        ratings, probs, confidences = wrapper.predict_rating(
            input_ids, attention_mask, sentiment_features
        )

        class_preds = ratings - 1  # Convert back to 0-9
        true_ratings = true_classes + 1  # Convert to 1-10

        all_ratings.extend(ratings)
        all_true_ratings.extend(true_ratings)
        all_class_preds.extend(class_preds)
        all_true_classes.extend(true_classes)

    # Calculate metrics
    accuracy = accuracy_score(all_true_classes, all_class_preds)
    unique_preds = np.unique(all_class_preds)
    
    print(f"Evaluation Results:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  Unique predictions: {len(unique_preds)}/10 classes")
    print(f"  Predicted classes: {unique_preds}")
    
    if len(unique_preds) <= 2:
        print("  WARNING: Model collapsed to binary predictions!")
    elif len(unique_preds) <= 5:
        print("  WARNING: Limited prediction diversity")
    else:
        print("  Good prediction diversity")
    
    return {
        'accuracy': accuracy,
        'unique_predictions': len(unique_preds),
        'predictions': all_ratings,
        'true_ratings': all_true_ratings
    }

print("Evaluation utilities defined")

In [ ]:
# Run evaluation
print("Running model evaluation...")
eval_results = evaluate_model(proto, drug_review_dm, device=proto.device)

print("\nEvaluation Summary:")
print(f"  Classification accuracy: {eval_results['accuracy']:.4f}")
print(f"  Prediction diversity: {eval_results['unique_predictions']}/10 classes")

if eval_results['accuracy'] > 0.2:
    print("Model is performing well!")
elif eval_results['accuracy'] > 0.1:
    print("Model is learning but needs improvement")
else:
    print("Model performance is poor - consider adjusting hyperparameters")

## Prototype Analysis

In [ ]:
# Analyze learned prototypes
import matplotlib.pyplot as plt

# Get prototype vectors
prototypes = proto.prototypes.detach().cpu().numpy()
print(f"Learned {prototypes.shape[0]} prototypes of dimension {prototypes.shape[1]}")

# Get sample representations
batch = next(iter(drug_review_dm.test_dataloader()))
with torch.no_grad():
    llm_out = proto.LLM(
        input_ids=batch['input_ids'].to(proto.device),
        attention_mask=batch['attention_mask'].to(proto.device),
        output_hidden_states=True
    )
    sample_reps = llm_out.hidden_states[-1][:, 0, :].cpu().numpy()

# Compute similarities between samples and prototypes
sample_vec = sample_reps[0]
proto_vecs = torch.tensor(prototypes)
similarities = F.cosine_similarity(torch.tensor(sample_vec).unsqueeze(0), proto_vecs)

# Plot sample-prototype similarities
plt.figure(figsize=(12, 6))
plt.bar(range(len(similarities)), similarities.numpy())
plt.xlabel("Prototype Index")
plt.ylabel("Cosine Similarity")
plt.title("Sample-Prototype Similarity Distribution")
plt.grid(True, alpha=0.3)
plt.show()

print(f"Max similarity: {similarities.max().item():.3f}")
print(f"Min similarity: {similarities.min().item():.3f}")
print(f"Mean similarity: {similarities.mean().item():.3f}")

## Model Saving

In [ ]:
# Save the trained model
import os
import json
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = f"proto_model_{timestamp}"
os.makedirs(save_dir, exist_ok=True)

print(f"Saving model to: {save_dir}")

# Save checkpoint
trainer.save_checkpoint(os.path.join(save_dir, "model_checkpoint.ckpt"))

# Save model weights
torch.save(proto.state_dict(), os.path.join(save_dir, "model_weights.pt"))

# Save tokenizer
tokenizer = drug_review_dm.tokenizer
tokenizer.save_pretrained(os.path.join(save_dir, "tokenizer"))

# Save configuration
config_data = {
    **args,
    "timestamp": timestamp,
    "model_type": "ProtoLM_classification"
}

with open(os.path.join(save_dir, "config.json"), "w") as f:
    json.dump(config_data, f, indent=2)

print("Model saved successfully!")
print(f"Contents: checkpoint, weights, tokenizer, config")